# Approach 2 / NB3 — Mispronunciation Detection (Whisper phoneme model)

Given a user recitation + the expected verse:
1. phonemize the expected diacritized verse -> canonical phonemes (MSA phonetiser),
2. Whisper-generate phonemes from the audio (the NB2 model),
3. align canonical vs predicted, **flag errors** + show the correct phoneme,
4. map errors back to words for the app UI,
5. (optional) report PER / F1 on QuranMB.v2.

Identical scoring logic to Approach 1's NB3 — only the recognizer differs
(Whisper .generate() instead of wav2vec2 CTC).

In [ ]:
!pip install -q "transformers==4.44.2" "datasets==2.21.0" soundfile librosa jiwer
!git clone -q https://github.com/Iqra-Eval/MSA_phonetiser.git || echo 'already cloned'

In [ ]:
import os, sys
import numpy as np
import torch

MODEL_DIR = "/kaggle/input/whisper-quran-phoneme/final"   # EDIT to your NB2 output
SAMPLING_RATE = 16_000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PHONETISER_DIR = "MSA_phonetiser"
print("device:", DEVICE, "| model:", MODEL_DIR)

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

model = WhisperForConditionalGeneration.from_pretrained(MODEL_DIR).to(DEVICE).eval()
processor = WhisperProcessor.from_pretrained(MODEL_DIR)

# Clear language/task forcing (the model emits phonemes; Tarteel's base config
# lacks lang_to_id, so forcing language="ar" in generate() crashes).
model.config.forced_decoder_ids = None
model.generation_config.forced_decoder_ids = None
model.generation_config.language = None
model.generation_config.task = None

def recognise_phonemes(audio_array):
    """audio (float32 @16k) -> predicted phoneme list via Whisper generate."""
    feats = processor.feature_extractor(np.asarray(audio_array, dtype=np.float32),
                                        sampling_rate=SAMPLING_RATE, return_tensors="pt")
    feats = feats.input_features.to(DEVICE)
    with torch.no_grad():
        # No language/task kwargs -> avoids the lang_to_id lookup crash.
        ids = model.generate(feats, max_length=448)
    text = processor.tokenizer.batch_decode(ids, skip_special_tokens=True)[0]
    return [t for t in text.split() if t]

In [ ]:
# ---- Canonical G2P: expected diacritized verse -> phoneme list ----------
sys.path.insert(0, PHONETISER_DIR)
_phonetise = None
try:
    from phonetiser import phonetise_Arabic
    _phonetise = phonetise_Arabic.phonetise
except Exception as e:
    print("phonetiser import failed:", e)

def canonical_phonemes(tashkeel_text):
    if _phonetise is None:
        raise RuntimeError("MSA phonetiser unavailable; check clone in cell 1.")
    _, with_spaces, _ = _phonetise(tashkeel_text)
    line = with_spaces[0] if isinstance(with_spaces, (list, tuple)) else with_spaces
    return [t for t in str(line).split() if t]

try:
    print(canonical_phonemes("بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ"))
except Exception as e:
    print("smoke test note:", e)

In [ ]:
# ---- Alignment + error tagging (same as Approach 1) ---------------------
def align(ref, hyp):
    n, m = len(ref), len(hyp)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): dp[i][0] = i
    for j in range(m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            c = 0 if ref[i-1] == hyp[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+c)
    i, j, ops = n, m, []
    while i > 0 or j > 0:
        if i>0 and j>0 and dp[i][j]==dp[i-1][j-1]+(0 if ref[i-1]==hyp[j-1] else 1):
            ops.append(("ok" if ref[i-1]==hyp[j-1] else "sub", ref[i-1], hyp[j-1])); i-=1; j-=1
        elif i>0 and dp[i][j]==dp[i-1][j]+1:
            ops.append(("del", ref[i-1], None)); i-=1
        else:
            ops.append(("ins", None, hyp[j-1])); j-=1
    return ops[::-1]

In [ ]:
# ---- Per-word verdicts (precision dial = MIN_ERRORS_PER_WORD) ------------
MIN_ERRORS_PER_WORD = 1

def analyse_recitation(audio_array, expected_tashkeel):
    words = expected_tashkeel.split()
    ref_flat, ref_word_of = [], []
    for wi, w in enumerate(words):
        ph = canonical_phonemes(w); ref_flat += ph; ref_word_of += [wi]*len(ph)
    hyp = recognise_phonemes(audio_array)
    ops = align(ref_flat, hyp)
    errors = {wi: [] for wi in range(len(words))}
    rp = 0
    for op, r, h in ops:
        if op in ("ok", "sub", "del"):
            wi = ref_word_of[rp] if rp < len(ref_word_of) else len(words)-1
            if op != "ok": errors[wi].append((op, r, h))
            rp += 1
        elif op == "ins":
            wi = ref_word_of[min(rp, len(ref_word_of)-1)] if ref_word_of else 0
            errors[wi].append((op, r, h))
    out = []
    for wi, w in enumerate(words):
        e = errors[wi]
        out.append({"word": w, "verdict": "flag" if len(e) >= MIN_ERRORS_PER_WORD else "pass",
                    "errors": e, "correct_phonemes": canonical_phonemes(w)})
    return out

In [ ]:
# ---- Demo on one clip ---------------------------------------------------
import soundfile as sf, subprocess
def load_audio(path):
    if path.endswith((".opus",".mp3",".m4a")):
        cmd=["ffmpeg","-loglevel","error","-i",path,"-f","f32le","-ar",str(SAMPLING_RATE),"-ac","1","pipe:1"]
        return np.frombuffer(subprocess.run(cmd, capture_output=True).stdout, dtype=np.float32).copy()
    a, _ = sf.read(path, dtype="float32"); return a

DEMO_AUDIO = "/kaggle/input/your-clip/fatiha.wav"   # EDIT
DEMO_VERSE = "بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ"   # EDIT
if os.path.exists(DEMO_AUDIO):
    for r in analyse_recitation(load_audio(DEMO_AUDIO), DEMO_VERSE):
        tag = "FLAG" if r["verdict"]=="flag" else "ok  "
        errs = "; ".join(f"{o}:{rr}->{hh}" for o,rr,hh in r["errors"]) or "-"
        print(f"[{tag}] {r['word']:15s} correct={' '.join(r['correct_phonemes'])}  errors={errs}")
else:
    print("Set DEMO_AUDIO to a real clip to run the demo.")

### Notes
- `MIN_ERRORS_PER_WORD` is the precision dial (raise it so the app flags less; fewer false alarms).
- For QuranMB.v2 detection F1 you need the gated GT phoneme annotations (accept terms on HF, set HF_TOKEN).
- If phonetiser symbols differ from what Whisper learned, add a small mapping table here.
- Compare this model's PER/F1 to the Approach-1 wav2vec2 model to decide which ships.